In [0]:
%run ../config

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# Ensure Silver Table Exists
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_catalog}.categories_clean (
    item_id INT NOT NULL,
    category1 STRING,
    category1_id STRING,
    category2 STRING,
    category2_id STRING,
    category3 STRING,
    category3_id STRING,
    category4 STRING,
    category4_id STRING,
    brand STRING,
    item_code INT,
    item_name STRING
)                   
""")


# Read Bronze table
df = spark.read.table(f"{bronze_catalog}.categories")

# Silver Transformation
df = (
    df
    # Standardize columns
    .withColumn("brand", F.initcap(F.col("brand")))
    .withColumn("item_name", F.initcap(F.col("item_name")))
)


# Final Schema
silver_df = (
    df.select(
        "item_id",
        "category1",
        "category1_id",
        "category2",
        "category2_id",
        "category3",
        "category3_id",
        "category4",
        "category4_id",
        "brand",
        "item_code",
        "item_name"
    )
)

# Write to silver table
(
silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{silver_catalog}.categories_clean")
)


# Validate that the Silver table was written successfully
count = spark.table(f"{silver_catalog}.categories_clean").count()

# Ensure table is not empty
assert count > 0, "categories_clean table is empty"

# Log result to job output
print(f"categories_clean validation passed: {count:,} records")